In [1]:
# Load the trained model and vectorizer
import lightgbm as lgb
import pickle

# Load the LightGBM model
model = lgb.Booster(model_file='price_prediction_model.txt')

# Load the TF-IDF vectorizer
with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

print("✅ Model and vectorizer loaded successfully!")
print(f"Model type: {type(model)}")
print(f"Vectorizer type: {type(vectorizer)}")

✅ Model and vectorizer loaded successfully!
Model type: <class 'lightgbm.basic.Booster'>
Vectorizer type: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [ ]:
# Alternative: Load with error handling
import lightgbm as lgb
import pickle
import os

try:
    # Check if files exist
    if not os.path.exists('price_prediction_model.txt'):
        raise FileNotFoundError("price_prediction_model.txt not found")
    if not os.path.exists('tfidf_vectorizer.pkl'):
        raise FileNotFoundError("tfidf_vectorizer.pkl not found")
    
    # Load the model and vectorizer
    model = lgb.Booster(model_file='price_prediction_model.txt')
    with open('tfidf_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    
    print("✅ Model and vectorizer loaded successfully!")
    print(f"Model features: {model.num_feature()}")
    print(f"Vectorizer vocabulary size: {len(vectorizer.vocabulary_)}")
    
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Please make sure you've run the training notebook and saved the model files.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

In [5]:
import os
import random
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb
import pickle
import re

def predictor(sample_id, catalog_content, image_link):
    '''
    Call your model/approach here
    
    Parameters:
    - sample_id: Unique identifier for the sample
    - catalog_content: Text containing product title and description
    - image_link: URL to product image
    
    Returns:
    - price: Predicted price as a float
    '''
    # Use the globally loaded model and vectorizer (from earlier cells)
    try:
        # Check if model and vectorizer are available
        if 'model' not in globals() or 'vectorizer' not in globals():
            return round(random.uniform(5.0, 500.0), 2)
    except:
        return round(random.uniform(5.0, 500.0), 2)
    
    def parse_catalog_content(content):
        if pd.isna(content) or content == '':
            return {'item_name': '', 'catalog_value': np.nan, 'catalog_unit': ''}
        parts = content.split('item_name:')
        if len(parts) < 2:
            return {'item_name': '', 'catalog_value': np.nan, 'catalog_unit': ''}
        item_section = parts[1].strip()
        item_name = item_section.split('|')[0].strip()
        value_pattern = r'(\d+(?:\.\d+)?)\s*([a-zA-Z]+(?:\s+[a-zA-Z]+)*)'
        matches = re.findall(value_pattern, content)
        if matches:
            value, unit = matches[0]
            return {'item_name': item_name, 'catalog_value': float(value), 'catalog_unit': unit.strip()}
        else:
            return {'item_name': item_name, 'catalog_value': np.nan, 'catalog_unit': ''}
    
    def standardize_unit(unit, value):
        if pd.isna(value) or unit == '':
            return value, ''
        unit_lower = unit.lower().strip()
        weight_conversions = {'oz': 28.3495, 'ounce': 28.3495, 'ounces': 28.3495, 'lb': 453.592, 'lbs': 453.592, 'pound': 453.592, 'pounds': 453.592, 'kg': 1000, 'kilogram': 1000, 'kilograms': 1000, 'g': 1, 'gram': 1, 'grams': 1, 'mg': 0.001, 'milligram': 0.001, 'milligrams': 0.001}
        volume_conversions = {'fl oz': 29.5735, 'fluid ounce': 29.5735, 'fluid ounces': 29.5735, 'cup': 236.588, 'cups': 236.588, 'pint': 473.176, 'pints': 473.176, 'quart': 946.353, 'quarts': 946.353, 'gallon': 3785.41, 'gallons': 3785.41, 'l': 1000, 'liter': 1000, 'liters': 1000, 'litre': 1000, 'litres': 1000, 'ml': 1, 'milliliter': 1, 'milliliters': 1, 'millilitre': 1, 'millilitres': 1}
        if unit_lower in weight_conversions:
            return value * weight_conversions[unit_lower], 'g'
        elif unit_lower in volume_conversions:
            return value * volume_conversions[unit_lower], 'ml'
        else:
            return value, unit_lower
    
    parsed = parse_catalog_content(catalog_content)
    item_name = parsed['item_name']
    catalog_value = parsed['catalog_value']
    catalog_unit = parsed['catalog_unit']
    standardized_value, standardized_unit = standardize_unit(catalog_unit, catalog_value)
    
    # Create text features - combine item_name and catalog_content like in training
    text_content = f"{catalog_content}".strip()  # Use full catalog_content for text features
    if text_content == '':
        text_content = 'unknown product'
    
    text_features = vectorizer.transform([text_content])
    text_features_dense = text_features.toarray()
    
    # Create the same features as used in training
    # Based on the training code, these features were used:
    # text_features + log_standardized_value + log_price_per_gram + log_price_per_ml + log_price_per_unit
    
    # Calculate log_standardized_value
    log_standardized_value = np.log1p(standardized_value) if not pd.isna(standardized_value) and standardized_value > 0 else 0
    
    # Calculate price per unit estimates (these were in training)
    # For prediction, we'll use reasonable defaults since we don't have actual price
    if standardized_unit == 'g' and standardized_value > 0:
        # Assume average price per gram for weight items
        log_price_per_gram = np.log1p(2.0)  # $2 per gram average
        log_price_per_ml = 0
        log_price_per_unit = np.log1p(standardized_value * 2.0)
    elif standardized_unit == 'ml' and standardized_value > 0:
        # Assume average price per ml for volume items  
        log_price_per_ml = np.log1p(0.5)  # $0.5 per ml average
        log_price_per_gram = 0
        log_price_per_unit = np.log1p(standardized_value * 0.5)
    else:
        log_price_per_gram = 0
        log_price_per_ml = 0
        log_price_per_unit = np.log1p(50)  # Default unit price
    
    # Build feature vector exactly like in training
    features = []
    features.extend(text_features_dense[0])  # Text features
    features.append(log_standardized_value)  # log_standardized_value
    features.append(log_price_per_gram)      # log_price_per_gram  
    features.append(log_price_per_ml)        # log_price_per_ml
    features.append(log_price_per_unit)      # log_price_per_unit
    
    features = np.array(features).reshape(1, -1)
    
    # Make prediction
    log_price_pred = model.predict(features)[0]
    price_pred = np.expm1(log_price_pred)
    
    # Ensure reasonable price bounds
    price_pred = max(1.0, min(price_pred, 1000.0))  # Between $1 and $1000
    
    return round(price_pred, 2)

if __name__ == "__main__":
    # DATASET_FOLDER = ''
    
    # Read test data
    test = pd.read_csv(os.path.join( 'test.csv'))
    
    # Apply predictor function to each row
    test['price'] = test.apply(
        lambda row: predictor(row['sample_id'], row['catalog_content'], row['image_link']), 
        axis=1
    )
    
    # Select only required columns for output
    output_df = test[['sample_id', 'price']]
    
    # Save predictions
    output_filename = os.path.join( 'test_out.csv')
    output_df.to_csv(output_filename, index=False)
    
    print(f"Predictions saved to {output_filename}")
    print(f"Total predictions: {len(output_df)}")
    print(f"Sample predictions:\n{output_df.head()}")

Predictions saved to test_out.csv
Total predictions: 75000
Sample predictions:
   sample_id  price
0     100179  73.24
1     245611  37.55
2     146263  47.11
3      95658  51.58
4      36806  71.42


In [2]:
import os
import random
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb
import pickle
import re

def predictor(sample_id, catalog_content, image_link):
    '''
    Call your model/approach here
    
    Parameters:
    - sample_id: Unique identifier for the sample
    - catalog_content: Text containing product title and description
    - image_link: URL to product image
    
    Returns:
    - price: Predicted price as a float
    '''
    # Load model and vectorizer
    try:
        model = lgb.Booster(model_file='price_prediction_model.txt')
        with open('tfidf_vectorizer.pkl', 'rb') as f:
            vectorizer = pickle.load(f)
    except:
        return round(random.uniform(5.0, 500.0), 2)
    
    def parse_catalog_content(content):
        if pd.isna(content) or content == '':
            return {'item_name': '', 'catalog_value': np.nan, 'catalog_unit': ''}
        parts = content.split('item_name:')
        if len(parts) < 2:
            return {'item_name': '', 'catalog_value': np.nan, 'catalog_unit': ''}
        item_section = parts[1].strip()
        item_name = item_section.split('|')[0].strip()
        value_pattern = r'(\d+(?:\.\d+)?)\s*([a-zA-Z]+(?:\s+[a-zA-Z]+)*)'
        matches = re.findall(value_pattern, content)
        if matches:
            value, unit = matches[0]
            return {'item_name': item_name, 'catalog_value': float(value), 'catalog_unit': unit.strip()}
        else:
            return {'item_name': item_name, 'catalog_value': np.nan, 'catalog_unit': ''}
    
    def standardize_unit(unit, value):
        if pd.isna(value) or unit == '':
            return value, ''
        unit_lower = unit.lower().strip()
        weight_conversions = {'oz': 28.3495, 'ounce': 28.3495, 'ounces': 28.3495, 'lb': 453.592, 'lbs': 453.592, 'pound': 453.592, 'pounds': 453.592, 'kg': 1000, 'kilogram': 1000, 'kilograms': 1000, 'g': 1, 'gram': 1, 'grams': 1, 'mg': 0.001, 'milligram': 0.001, 'milligrams': 0.001}
        volume_conversions = {'fl oz': 29.5735, 'fluid ounce': 29.5735, 'fluid ounces': 29.5735, 'cup': 236.588, 'cups': 236.588, 'pint': 473.176, 'pints': 473.176, 'quart': 946.353, 'quarts': 946.353, 'gallon': 3785.41, 'gallons': 3785.41, 'l': 1000, 'liter': 1000, 'liters': 1000, 'litre': 1000, 'litres': 1000, 'ml': 1, 'milliliter': 1, 'milliliters': 1, 'millilitre': 1, 'millilitres': 1}
        if unit_lower in weight_conversions:
            return value * weight_conversions[unit_lower], 'g'
        elif unit_lower in volume_conversions:
            return value * volume_conversions[unit_lower], 'ml'
        else:
            return value, unit_lower
    
    parsed = parse_catalog_content(catalog_content)
    item_name = parsed['item_name']
    catalog_value = parsed['catalog_value']
    catalog_unit = parsed['catalog_unit']
    standardized_value, standardized_unit = standardize_unit(catalog_unit, catalog_value)
    
    # Create text features - combine item_name and catalog_content like in training
    text_content = f"{catalog_content}".strip()  # Use full catalog_content for text features
    if text_content == '':
        text_content = 'unknown product'
    
    text_features = vectorizer.transform([text_content])
    text_features_dense = text_features.toarray()
    
    # Create features to match training (without price-per-unit circular logic)
    # Use only: text_features + log_standardized_value + measurement_type indicators
    
    # Calculate log_standardized_value (like in training)
    log_standardized_value = np.log1p(standardized_value) if not pd.isna(standardized_value) and standardized_value > 0 else 0
    
    # Create measurement type indicators (like categorical encoding)
    measurement_type = 'weight' if standardized_unit == 'g' else 'volume' if standardized_unit == 'ml' else 'other'
    is_weight = 1 if measurement_type == 'weight' else 0
    is_volume = 1 if measurement_type == 'volume' else 0
    is_other = 1 if measurement_type == 'other' else 0
    
    # Build feature vector
    features = []
    features.extend(text_features_dense[0])  # Text features (TF-IDF)
    features.append(log_standardized_value)  # Log of standardized value
    features.append(is_weight)               # Weight indicator
    features.append(is_volume)               # Volume indicator  
    features.append(is_other)                # Other measurement indicator
    
    features = np.array(features).reshape(1, -1)
    
    # Make prediction
    try:
        prediction = model.predict(features)
        # Convert to numpy array and extract scalar
        pred_array = np.asarray(prediction).flatten()
        log_price_pred = pred_array[0]
        
        # Convert from log scale back to actual price
        price_pred = np.expm1(log_price_pred)
        
        # Ensure reasonable price bounds
        price_pred = max(1.0, min(price_pred, 1000.0))
        
        return round(price_pred, 2)
    except Exception as e:
        # Fallback to random if prediction fails
        print(f"Prediction error: {e}")
        return round(random.uniform(10.0, 200.0), 2)

if __name__ == "__main__":
    # DATASET_FOLDER = ''
    
    # Read test data
    test = pd.read_csv(os.path.join( 'test.csv'))
    
    # Apply predictor function to each row
    test['price'] = test.apply(
        lambda row: predictor(row['sample_id'], row['catalog_content'], row['image_link']), 
        axis=1
    )
    
    # Select only required columns for output
    output_df = test[['sample_id', 'price']]
    
    # Save predictions
    output_filename = os.path.join( 'test_out1.csv')
    output_df.to_csv(output_filename, index=False)
    
    print(f"Predictions saved to {output_filename}")
    print(f"Total predictions: {len(output_df)}")
    print(f"Sample predictions:\n{output_df.head()}")

Predictions saved to test_out1.csv
Total predictions: 75000
Sample predictions:
   sample_id  price
0     100179   5.22
1     245611   1.88
2     146263   2.79
3      95658   2.52
4      36806   4.85
